In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- s8_combine_load_concat_write ---
FIX_S8_COMBINE_LOAD_CONCAT_WRITE_DF_LIST_BEFORE = [pd.DataFrame({"id":[1,2,3],"value":[10,20,30]}), pd.DataFrame({"id":[4,5,6],"value":[40,50,60]})]
FIX_S8_COMBINE_LOAD_CONCAT_WRITE_DF_LIST_GEN = [pl.DataFrame({"id":[1,2,3],"value":[10,20,30]}), pl.DataFrame({"id":[4,5,6],"value":[40,50,60]})]
FIX_S8_COMBINE_LOAD_CONCAT_WRITE_FILEPATH = "test_file.csv"
FIX_S8_COMBINE_LOAD_CONCAT_WRITE_LOAD_DF_FROM_PICKLE = lambda path: pd.DataFrame({"id":[1,2,3],"title":["A","B","C"],"label":[0,1,0]})
FIX_S8_COMBINE_LOAD_CONCAT_WRITE_PATH_DOCUMENTS_AUTHORS_LABELS_CITATIONS = "test_file.csv"
FIX_S8_COMBINE_LOAD_CONCAT_WRITE_WRITE_DF_TO_PICKLE = lambda df, path: None  # stub

print("✅ Fixtures loaded")

In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_s8_combine_load_concat_write(df_list, filepath, load_df_from_pickle, path_documents_authors_labels_citations, write_df_to_pickle):
    df_chunk = load_df_from_pickle(filepath)
    ...
    df_combined = pd.concat(df_list)
    ...
    write_df_to_pickle(df_combined, path_documents_authors_labels_citations)
    return df_combined

In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_s8_combine_load_concat_write(df_list, filepath, load_df_from_pickle, path_documents_authors_labels_citations, write_df_to_pickle):
    df_chunk = load_df_from_pickle(filepath)
    ...
    df_combined = pl.concat(df_list)
    ...
    write_df_to_pickle(df_combined, path_documents_authors_labels_citations)
    return df_combined

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: s8_combine_load_concat_write ===

# L1 smoke – generated
try:
    _r = gen_s8_combine_load_concat_write(FIX_S8_COMBINE_LOAD_CONCAT_WRITE_DF_LIST_GEN, FIX_S8_COMBINE_LOAD_CONCAT_WRITE_FILEPATH, FIX_S8_COMBINE_LOAD_CONCAT_WRITE_LOAD_DF_FROM_PICKLE, FIX_S8_COMBINE_LOAD_CONCAT_WRITE_PATH_DOCUMENTS_AUTHORS_LABELS_CITATIONS, FIX_S8_COMBINE_LOAD_CONCAT_WRITE_WRITE_DF_TO_PICKLE)
    print("✅ L1 smoke gen_s8_combine_load_concat_write: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_s8_combine_load_concat_write: {type(_e).__name__}: {_e}")

# L1 smoke – before
try:
    _rb = before_s8_combine_load_concat_write(FIX_S8_COMBINE_LOAD_CONCAT_WRITE_DF_LIST_BEFORE, FIX_S8_COMBINE_LOAD_CONCAT_WRITE_FILEPATH, FIX_S8_COMBINE_LOAD_CONCAT_WRITE_LOAD_DF_FROM_PICKLE, FIX_S8_COMBINE_LOAD_CONCAT_WRITE_PATH_DOCUMENTS_AUTHORS_LABELS_CITATIONS, FIX_S8_COMBINE_LOAD_CONCAT_WRITE_WRITE_DF_TO_PICKLE)
    print("✅ L1 smoke before_s8_combine_load_concat_write: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_s8_combine_load_concat_write: {type(_e).__name__}: {_e}")

# L2 behavioral equivalence
try:
    _rb = before_s8_combine_load_concat_write(FIX_S8_COMBINE_LOAD_CONCAT_WRITE_DF_LIST_BEFORE, FIX_S8_COMBINE_LOAD_CONCAT_WRITE_FILEPATH, FIX_S8_COMBINE_LOAD_CONCAT_WRITE_LOAD_DF_FROM_PICKLE, FIX_S8_COMBINE_LOAD_CONCAT_WRITE_PATH_DOCUMENTS_AUTHORS_LABELS_CITATIONS, FIX_S8_COMBINE_LOAD_CONCAT_WRITE_WRITE_DF_TO_PICKLE)
    _rg = gen_s8_combine_load_concat_write(FIX_S8_COMBINE_LOAD_CONCAT_WRITE_DF_LIST_GEN, FIX_S8_COMBINE_LOAD_CONCAT_WRITE_FILEPATH, FIX_S8_COMBINE_LOAD_CONCAT_WRITE_LOAD_DF_FROM_PICKLE, FIX_S8_COMBINE_LOAD_CONCAT_WRITE_PATH_DOCUMENTS_AUTHORS_LABELS_CITATIONS, FIX_S8_COMBINE_LOAD_CONCAT_WRITE_WRITE_DF_TO_PICKLE)
    compare(_rb, _rg, "s8_combine_load_concat_write")
except Exception as _e:
    print(f"❌ L2 equivalence s8_combine_load_concat_write: setup error — {type(_e).__name__}: {_e}")

# L3 edge - concatenate one schema-bearing empty chunk.
try:
    _empty_pd = [pd.DataFrame({
        "id": pd.Series(dtype="int64"),
        "value": pd.Series(dtype="int64"),
    })]
    _empty_pl = [pl.DataFrame(schema={"id": pl.Int64, "value": pl.Int64})]
    _rb = before_s8_combine_load_concat_write(
        _empty_pd,
        FIX_S8_COMBINE_LOAD_CONCAT_WRITE_FILEPATH,
        FIX_S8_COMBINE_LOAD_CONCAT_WRITE_LOAD_DF_FROM_PICKLE,
        FIX_S8_COMBINE_LOAD_CONCAT_WRITE_PATH_DOCUMENTS_AUTHORS_LABELS_CITATIONS,
        FIX_S8_COMBINE_LOAD_CONCAT_WRITE_WRITE_DF_TO_PICKLE,
    )
    _rg = gen_s8_combine_load_concat_write(
        _empty_pl,
        FIX_S8_COMBINE_LOAD_CONCAT_WRITE_FILEPATH,
        FIX_S8_COMBINE_LOAD_CONCAT_WRITE_LOAD_DF_FROM_PICKLE,
        FIX_S8_COMBINE_LOAD_CONCAT_WRITE_PATH_DOCUMENTS_AUTHORS_LABELS_CITATIONS,
        FIX_S8_COMBINE_LOAD_CONCAT_WRITE_WRITE_DF_TO_PICKLE,
    )
    compare(_rb, _rg, "L3 edge s8_combine_load_concat_write empty chunk", check_row_order=True)
except Exception as _e:
    print(f"❌ L3 edge s8_combine_load_concat_write empty chunk: {type(_e).__name__}: {_e}")

# AUDIT-S8: verify loader invocation and writer side effect.
try:
    _bcalls={"load":[],"write":[]}; _gcalls={"load":[],"write":[]}
    def _bl(path): _bcalls["load"].append(path); return pd.DataFrame({"id":[1]})
    def _gl(path): _gcalls["load"].append(path); return pl.DataFrame({"id":[1]})
    def _bw(frame,path): _bcalls["write"].append((frame.copy(),path))
    def _gw(frame,path): _gcalls["write"].append((frame.clone(),path))
    _rb=before_s8_combine_load_concat_write(FIX_S8_COMBINE_LOAD_CONCAT_WRITE_DF_LIST_BEFORE,FIX_S8_COMBINE_LOAD_CONCAT_WRITE_FILEPATH,_bl,FIX_S8_COMBINE_LOAD_CONCAT_WRITE_PATH_DOCUMENTS_AUTHORS_LABELS_CITATIONS,_bw)
    _rg=gen_s8_combine_load_concat_write(FIX_S8_COMBINE_LOAD_CONCAT_WRITE_DF_LIST_GEN,FIX_S8_COMBINE_LOAD_CONCAT_WRITE_FILEPATH,_gl,FIX_S8_COMBINE_LOAD_CONCAT_WRITE_PATH_DOCUMENTS_AUTHORS_LABELS_CITATIONS,_gw)
    if _bcalls["load"]==_gcalls["load"] and _bcalls["write"][0][1]==_gcalls["write"][0][1]: print("✅ L2 equivalence s8_combine_load_concat_write calls: MATCH")
    else: print(f"❌ L2 equivalence s8_combine_load_concat_write calls: MISMATCH — before={_bcalls}, gen={_gcalls}")
    compare(_bcalls["write"][0][0],_gcalls["write"][0][0],"s8_combine_load_concat_write writer frame",check_row_order=True)
except Exception as _e: print(f"❌ L2 equivalence s8_combine_load_concat_write side effects: {type(_e).__name__}: {_e}")
